In [0]:
CATALOG = "workspace"

RAW_PATH = (
    f"/Volumes/{CATALOG}/raw_data/ecommerce_files/"
)

TABLES = [
    "customers",
    "orders",
    "order_items",
    "products",
    "payments"
]

In [0]:

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{RAW_PATH}orders.csv")
)

display(orders_df)


order_id,customer_id,order_date,status,channel,subtotal,discount_amount,shipping_fee,tax_amount,total_amount,currency
1,1,2026-04-18 22:37:25,delivered,app,14.29,0,7.73,1.14,23.16,USD
2,1,2025-06-09 08:05:40,delivered,marketplace,630.95,0,0,50.48,681.43,USD
3,1,2024-11-29 08:01:39,delivered,app,345.06,62.79,0,22.58,304.85,USD
4,1,2024-11-19 20:11:54,delivered,web,431.38,0,0,34.51,465.89,USD
5,2,2025-12-28 04:22:31,delivered,web,613.4,0,0,49.07,662.47,USD
6,2,2024-04-30 16:29:23,delivered,marketplace,917.21,0,0,73.38,990.59,USD
7,2,2026-07-09 21:08:55,delivered,marketplace,2019.54,0,0,161.56,2181.1,USD
8,2,2024-11-02 14:14:42,delivered,app,438.11,0,0,35.05,473.16,USD
9,3,2024-07-20 00:47:06,delivered,app,2222.19,0,0,177.78,2399.97,USD
10,3,2024-01-09 02:18:16,delivered,app,813.91,0,0,65.11,879.02,USD


In [0]:

orders_df.printSchema()
print("Rows:", orders_df.count())
display(orders_df.limit(10))


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- shipping_fee: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- currency: string (nullable = true)

Rows: 90000


order_id,customer_id,order_date,status,channel,subtotal,discount_amount,shipping_fee,tax_amount,total_amount,currency
1,1,2026-04-18 22:37:25,delivered,app,14.29,0,7.73,1.14,23.16,USD
2,1,2025-06-09 08:05:40,delivered,marketplace,630.95,0,0,50.48,681.43,USD
3,1,2024-11-29 08:01:39,delivered,app,345.06,62.79,0,22.58,304.85,USD
4,1,2024-11-19 20:11:54,delivered,web,431.38,0,0,34.51,465.89,USD
5,2,2025-12-28 04:22:31,delivered,web,613.4,0,0,49.07,662.47,USD
6,2,2024-04-30 16:29:23,delivered,marketplace,917.21,0,0,73.38,990.59,USD
7,2,2026-07-09 21:08:55,delivered,marketplace,2019.54,0,0,161.56,2181.1,USD
8,2,2024-11-02 14:14:42,delivered,app,438.11,0,0,35.05,473.16,USD
9,3,2024-07-20 00:47:06,delivered,app,2222.19,0,0,177.78,2399.97,USD
10,3,2024-01-09 02:18:16,delivered,app,813.91,0,0,65.11,879.02,USD


In [0]:

from pyspark.sql import functions as F

for table in TABLES:

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(f"{RAW_PATH}{table}.csv")
    )

    df = (
        df.withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.lit(f"{table}.csv")
        )
    )

    target = f"{CATALOG}.bronze.{table}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"{target}: {df.count()} rows")


workspace.bronze.customers: 26000 rows
workspace.bronze.orders: 90000 rows
workspace.bronze.order_items: 200000 rows
workspace.bronze.products: 1500 rows
workspace.bronze.payments: 90000 rows


In [0]:
%sql
SELECT * FROM workspace.bronze.orders LIMIT 10;

order_id,customer_id,order_date,status,channel,subtotal,discount_amount,shipping_fee,tax_amount,total_amount,currency,_ingested_at,_source_file
1,1,2026-04-18 22:37:25,delivered,app,14.29,0,7.73,1.14,23.16,USD,2026-11-08T03:29:24.044Z,orders.csv
2,1,2025-06-09 08:05:40,delivered,marketplace,630.95,0,0,50.48,681.43,USD,2026-11-08T03:29:24.044Z,orders.csv
3,1,2024-11-29 08:01:39,delivered,app,345.06,62.79,0,22.58,304.85,USD,2026-11-08T03:29:24.044Z,orders.csv
4,1,2024-11-19 20:11:54,delivered,web,431.38,0,0,34.51,465.89,USD,2026-11-08T03:29:24.044Z,orders.csv
5,2,2025-12-28 04:22:31,delivered,web,613.4,0,0,49.07,662.47,USD,2026-11-08T03:29:24.044Z,orders.csv
6,2,2024-04-30 16:29:23,delivered,marketplace,917.21,0,0,73.38,990.59,USD,2026-11-08T03:29:24.044Z,orders.csv
7,2,2026-07-09 21:08:55,delivered,marketplace,2019.54,0,0,161.56,2181.1,USD,2026-11-08T03:29:24.044Z,orders.csv
8,2,2024-11-02 14:14:42,delivered,app,438.11,0,0,35.05,473.16,USD,2026-11-08T03:29:24.044Z,orders.csv
9,3,2024-07-20 00:47:06,delivered,app,2222.19,0,0,177.78,2399.97,USD,2026-11-08T03:29:24.044Z,orders.csv
10,3,2024-01-09 02:18:16,delivered,app,813.91,0,0,65.11,879.02,USD,2026-11-08T03:29:24.044Z,orders.csv


In [0]:
%sql
SHOW TABLES IN workspace.bronze;

database,tableName,isTemporary
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,payments,false
bronze,products,false


In [0]:
%sql
SELECT SUM(row_count) AS total_raw_records
FROM (
    SELECT COUNT(*) AS row_count FROM bronze.customers
    UNION ALL
    SELECT COUNT(*) FROM bronze.orders
    UNION ALL
    SELECT COUNT(*) FROM bronze.order_items
    UNION ALL
    SELECT COUNT(*) FROM bronze.payments
    UNION ALL
    SELECT COUNT(*) FROM bronze.products
);

total_raw_records
407500
